In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# LAMMPS LOG PARSER
# ============================================================

def extract_thermo_data(
    log_path,
    cutoff_step=None,
    last_ns=None,
    timestep_fs=1.0,
):
    """
    Extract thermo quantities from a LAMMPS log file.

    Parameters
    ----------
    log_path : str
        Path to log.lammps

    cutoff_step : int or None
        Use data after this step only

    last_ns : float or None
        Extract only last X ns

    timestep_fs : float
        MD timestep in fs

    Returns
    -------
    thermo_dict : dict
        Dictionary containing all thermo quantities
    """

    with open(log_path, "r") as f:
        lines = f.readlines()

    # --------------------------------------------------------
    # Find thermo header
    # --------------------------------------------------------

    header = None
    start_idx = None

    for i, line in enumerate(lines):
        if line.strip().startswith("Step"):
            cols = line.split()

            if "PotEng" in cols or "pe" in cols:
                header = cols
                start_idx = i
                break

    if header is None:
        raise RuntimeError(f"Thermo header not found in {log_path}")

    # --------------------------------------------------------
    # Store thermo data
    # --------------------------------------------------------

    thermo = {key: [] for key in header}

    for line in lines[start_idx + 1:]:

        parts = line.split()

        if len(parts) != len(header):
            continue

        try:
            step = int(parts[0])
        except:
            continue

        for key, value in zip(header, parts):

            try:
                thermo[key].append(float(value))
            except:
                pass

    # Convert to numpy arrays
    for key in thermo:
        thermo[key] = np.array(thermo[key])

    # --------------------------------------------------------
    # Filtering
    # --------------------------------------------------------

    steps = thermo["Step"]

    if cutoff_step is not None:

        mask = steps >= cutoff_step

    elif last_ns is not None:

        max_step = steps.max()

        steps_target = int((last_ns * 1e6) / timestep_fs)

        cutoff = max_step - steps_target

        mask = steps >= cutoff

    else:

        mask = np.ones_like(steps, dtype=bool)

    for key in thermo:
        thermo[key] = thermo[key][mask]

    return thermo


# ============================================================
# ENERGY ANALYSIS
# ============================================================

def compute_adsorption_energy(
    close_log,
    peg_log,
    csh_log,
    cutoff_step=None,
    average_last_n=None,
    two_box=False,
):
    """
    Compute adsorption energy:

    E_ads = E_close - (E_peg + E_csh)
    """

    close_data = extract_thermo_data(
        close_log,
        cutoff_step=cutoff_step,
    )

    peg_data = extract_thermo_data(
        peg_log,
        cutoff_step=cutoff_step,
    )

    csh_data = extract_thermo_data(
        csh_log,
        cutoff_step=cutoff_step,
    )

    # --------------------------------------------------------
    # Detect PE column
    # --------------------------------------------------------

    pe_key = "PotEng" if "PotEng" in close_data else "pe"

    close_pe = close_data[pe_key]
    peg_pe = peg_data[pe_key]
    csh_pe = csh_data[pe_key]

    # --------------------------------------------------------
    # Average strategy
    # --------------------------------------------------------

    if average_last_n is not None:

        E_close = np.mean(close_pe[-average_last_n:])
        E_peg = np.mean(peg_pe[-average_last_n:])
        E_csh = np.mean(csh_pe[-average_last_n:])

    else:

        E_close = np.mean(close_pe)
        E_peg = np.mean(peg_pe)
        E_csh = np.mean(csh_pe)

    if two_box:
        E_ads = E_close - E_peg
    else:
        E_ads = E_close - (E_peg + E_csh)

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print("\n==============================")
    print("Adsorption Energy Summary")
    print("==============================")

    print(f"Close System     : {E_close:.6f}")
    print(f"PEG + Water      : {E_peg:.6f}")
    print(f"CSH + Water      : {E_csh:.6f}")
    print(f"Separated System : {E_peg + E_csh:.6f}")

    print("------------------------------")
    print(f"Adsorption Energy: {E_ads:.6f} kcal/mol")
    print("==============================\n")

    return {
        "close": close_data,
        "peg": peg_data,
        "csh": csh_data,
        "E_ads": E_ads,
    }


# ============================================================
# PLOTTING
# ============================================================

def plot_thermo_grid(data_dict):

    quantities = [
        "Temp",
        "PotEng",
        "TotEng",
        "Press",
    ]

    systems = [
        ("close", "Close"),
        ("peg", "PEG + Water"),
        ("csh", "CSH + Water"),
    ]

    fig, axs = plt.subplots(
        len(quantities),
        len(systems),
        figsize=(15, 12),
    )

    for row, quantity in enumerate(quantities):

        for col, (key, title) in enumerate(systems):

            ax = axs[row, col]

            data = data_dict[key]

            if quantity not in data:
                ax.set_visible(False)
                continue

            ax.plot(data[quantity])

            ax.set_title(f"{title} : {quantity}")

            ax.set_xlabel("Frame")
            ax.set_ylabel(quantity)

    plt.tight_layout()
    plt.show()


def plot_thermo_grid_together(data_dict, quantities, systems, fig, axs, name=None):

    for row, quantity in enumerate(quantities):

        for col, (key, title) in enumerate(systems):

            ax = axs[row, col]

            data = data_dict[key]

            if quantity not in data:
                ax.set_visible(False)
                continue

            ax.plot(data[quantity], label=name, alpha=0.5)

            ax.set_title(f"{title} : {quantity}")

            ax.legend()
            ax.set_xlabel("Frame")
            ax.set_ylabel(quantity)

# ============================================================
# USER INPUT
# ============================================================

BASE = Path(
    ""
    "MDSetup/example/mechanical_properties/"
    "0_CSH_transfer/CSH_surface/try/"
    "2_combined_new"
)

# RUN = "0_run11"

# close_log = BASE / RUN / "1_all3/log.lammps"
# peg_log   = BASE / RUN / "2_peg/log.lammps"
# csh_log   = BASE / RUN / "3_csh_water/log.lammps"


In [ ]:
RUN = "0_run11"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
pe_npt1 = results_11n['csh']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_21n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31n['peg']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_11n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_21n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")

pe_npt1 = results_31n['close']['PotEng'][-1000:]
z = np.polyfit(np.arange(len(pe_npt1)), pe_npt1, 1)
print(f"Slope of PE vs time: {z[0]:.6f} kcal/mol")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 3000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 3000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # Piecewise linear fits        
    slopes = []
    for start in range(0, len(pe) - fit_window + 1, fit_window):
        xs = x[start:start + fit_window]
        ys = pe[start:start + fit_window]
        z = np.polyfit(xs, ys, 1)
        p = np.poly1d(z)
        slopes.append(z[0])
        ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ###### last 10
# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038016.780000
# PEG + Water      : -72363.113200
# CSH + Water      : -965747.788000
# Separated System : -1038110.901200
# ------------------------------
# Adsorption Energy: 94.121200 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040272.210000
# PEG + Water      : -74615.443400
# CSH + Water      : -965747.788000
# Separated System : -1040363.231400
# ------------------------------
# Adsorption Energy: 91.021400 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041700.290000
# PEG + Water      : -75692.293600
# CSH + Water      : -965747.788000
# Separated System : -1041440.081600
# ------------------------------
# Adsorption Energy: -260.208400 kcal/mol
# ==============================




# ###### last 100

# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038137.302000
# PEG + Water      : -72349.151700
# CSH + Water      : -965753.746000
# Separated System : -1038102.897700
# ------------------------------
# Adsorption Energy: -34.404300 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040214.262000
# PEG + Water      : -74594.949120
# CSH + Water      : -965753.746000
# Separated System : -1040348.695120
# ------------------------------
# Adsorption Energy: 134.433120 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041625.384000
# PEG + Water      : -75731.135010
# CSH + Water      : -965753.746000
# Separated System : -1041484.881010
# ------------------------------
# Adsorption Energy: -140.502990 kcal/mol
# ==============================


# # last 500
# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038091.537400
# PEG + Water      : -72346.851272
# CSH + Water      : -965732.518540
# Separated System : -1038079.369812
# ------------------------------
# Adsorption Energy: -12.167588 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040158.576800
# PEG + Water      : -74600.541232
# CSH + Water      : -965732.518540
# Separated System : -1040333.059772
# ------------------------------
# Adsorption Energy: 174.482972 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041604.613400
# PEG + Water      : -75721.612336
# CSH + Water      : -965732.518540
# Separated System : -1041454.130876
# ------------------------------
# Adsorption Energy: -150.482524 kcal/mol
# ==============================



# # last 1000

# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1038052.625300
# PEG + Water      : -72349.848263
# CSH + Water      : -965703.972920
# Separated System : -1038053.821183
# ------------------------------
# Adsorption Energy: 1.195883 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1040126.716600
# PEG + Water      : -74600.893075
# CSH + Water      : -965703.972920
# Separated System : -1040304.865995
# ------------------------------
# Adsorption Energy: 178.149395 kcal/mol
# ==============================


# ==============================
# Adsorption Energy Summary
# ==============================
# Close System     : -1041572.457100
# PEG + Water      : -75726.672740
# CSH + Water      : -965703.972920
# Separated System : -1041430.645660
# ------------------------------
# Adsorption Energy: -141.811440 kcal/mol
# ==============================

In [ ]:
RUN = "0_run12_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 2000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 500       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 7000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # # Piecewise linear fits        
    # slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 7000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # Piecewise linear fits        
    slopes = []
    for start in range(0, len(pe) - fit_window + 1, fit_window):
        xs = x[start:start + fit_window]
        ys = pe[start:start + fit_window]
        z = np.polyfit(xs, ys, 1)
        p = np.poly1d(z)
        slopes.append(z[0])
        ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run12_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log_full.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/log_full.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=5000)


# results_11n['peg'] has total length of 7000 while others are 12000 so we need to append the average of all property from last 1000 steps and make it 12000
for key in results_11n['peg'].keys():
    avg_value = np.mean(results_11n['peg'][key][-1000:])
    padding = np.full(5000, avg_value)
    results_11n['peg'][key] = np.concatenate([results_11n['peg'][key], padding])


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))

plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 9000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 3000       # local linear fit window

systems = [("1:1", results_11n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[j]
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        ax.set_xlabel("Time (steps)")
        ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 11000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 1, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    # ax = axes[i]
    ax = axes
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # Piecewise linear fits        
    slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)
    ax.set_ylim([-1000,1000])
    plt.axhline(0)


# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run13_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


avg_peg11_ = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21_ = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31_ = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11_:.4f}, 2:1: {avg_peg21_:.4f}, 3:1: {avg_peg31_:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 2000          # number of points from the end
smooth_window = 20    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()



In [ ]:
systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

for i, (ratio, results) in enumerate(systems):    
    # Data
    print(len(np.asarray(results[phases[0]]["PotEng"])))

In [ ]:
pe2 = pe2_list_[i]
pe2

In [ ]:
len(results[phases[0]]["PotEng"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 5000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]


pe2_list_ = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    len_ = len(np.asarray(results[phases[0]]["PotEng"]))
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][len_-nsteps:len_])
    pe2 = pe2_list_[i]
    pe3 = np.asarray(results[phases[2]]["PotEng"][len_-nsteps:len_])
    
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # # Piecewise linear fits        
    # slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 5000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]


pe2_list_ = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    len_ = len(np.asarray(results[phases[0]]["PotEng"]))
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = pe2_list_[i]
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # # Piecewise linear fits        
    # slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run13_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log_re.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log_re.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log_re.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()

avg_peg11 = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21 = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31 = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11:.4f}, 2:1: {avg_peg21:.4f}, 3:1: {avg_peg31:.4f}")


In [ ]:
# print(-73367.7947+72349.8483)
# print(-75639.7142+74600.8931)
# print(-76757.8462+75726.6727)

In [ ]:
# plt pot energy of peg only.

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# Settings
smooth_window = 1
frac = 0.20          # 20% of data used for each local regression

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)
nsteps = 1000
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    pe1 = np.asarray(results["close"]["PotEng"][-nsteps:])
    pe2 = np.asarray(results["peg"]["PotEng"][-nsteps:])
    # pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][-nsteps:])
    pe = pe2
    x = np.arange(len(pe))

    ax.plot(x, pe, color='red', alpha=0.35, lw=1, label='Raw trajectory')

    # LOWESS trend    
    trend = lowess(pe, x, frac=frac, return_sorted=True)

    ax.plot(trend[:,0], trend[:,1], 'k--', lw=3, label='LOWESS trend')

    # Overall slope of LOWESS trend
    slope, intercept = np.polyfit(trend[:,0], trend[:,1], 1)
    # plot pe2_list[i]
    ax.plot(x, np.full_like(x, pe2_list[i]), 'b--', lw=2, label='Average PotEng for PEG phase')
    ax.set_title(f"({ratio})\nOverall slope = {slope:.5f}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("pe2 (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# print(len(results_11n['close']['Step']), len(results_11n['peg']['Step']), len(results_11n['csh']['Step']))
# # (13770, 5106, 12024)
# print(len(results_21n['close']['Step']), len(results_21n['peg']['Step']), len(results_21n['csh']['Step']))
# # (12724, 5406, 12024)
# print(len(results_31n['close']['Step']), len(results_31n['peg']['Step']), len(results_31n['csh']['Step']))
# # (13906, 5106, 12024)

# need to make all three lenght equal to 13770 by appending the average of last 1000 steps for peg and for csh
for key in results_11n['peg'].keys():
    avg_value = np.mean(results_11n['peg'][key][-1000:])
    padding = np.full(8664, avg_value)
    results_11n['peg'][key] = np.concatenate([results_11n['peg'][key], padding])
for key in results_11n['csh'].keys():
    avg_value = np.mean(results_11n['csh'][key][-1000:])
    padding = np.full(1746, avg_value)
    results_11n['csh'][key] = np.concatenate([results_11n['csh'][key], padding])


# need to make all three lenght equal to 13770 by appending the average of last 1000 steps for peg and for csh
for key in results_21n['peg'].keys():
    avg_value = np.mean(results_21n['peg'][key][-1000:])
    padding = np.full(7318, avg_value)
    results_21n['peg'][key] = np.concatenate([results_21n['peg'][key], padding])
for key in results_21n['csh'].keys():
    avg_value = np.mean(results_21n['csh'][key][-1000:])
    padding = np.full(700, avg_value)
    results_21n['csh'][key] = np.concatenate([results_21n['csh'][key], padding])


# need to make all three lenght equal to 13770 by appending the average of last 1000 steps for peg and for csh
for key in results_31n['peg'].keys():
    avg_value = np.mean(results_31n['peg'][key][-1000:])
    padding = np.full(8800, avg_value)
    results_31n['peg'][key] = np.concatenate([results_31n['peg'][key], padding])
for key in results_31n['csh'].keys():
    avg_value = np.mean(results_31n['csh'][key][-1000:])
    padding = np.full(1882, avg_value)
    results_31n['csh'][key] = np.concatenate([results_31n['csh'][key], padding])

In [ ]:
quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 4000          # number of points from the end
smooth_window = 20    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 9000          # number of points from the end
smooth_window = 500    # moving average window
fit_window = 1000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][-nsteps:])
    pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][-nsteps:])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

    # # Piecewise linear fits        
    # slopes = []
    # for start in range(0, len(pe) - fit_window + 1, fit_window):
    #     xs = x[start:start + fit_window]
    #     ys = pe[start:start + fit_window]
    #     z = np.polyfit(xs, ys, 1)
    #     p = np.poly1d(z)
    #     slopes.append(z[0])
    #     ax.plot(xs, p(xs), 'k--', lw=2)

    # # Summary statistics        
    # avg_slope = np.mean(slopes)
    # final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    # slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    slope_text = ''
    ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
#####restart2



RUN = "0_run13_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/restart2/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/restart/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/restart2/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/restart/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/restart2/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/restart/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 5420          # number of points from the end
smooth_window = 1    # moving average window
fit_window = 2000       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase
# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    nsteps = len(results[phases[0]]["PotEng"])  # Use the length of the "close" phase for nsteps
    # Data
    pe1 = np.asarray(results[phases[0]]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    # pe2 = np.asarray(results[phases[1]]["PotEng"][-nsteps:])
    pe3 = np.asarray(results[phases[2]]["PotEng"][:nsteps])
    pe_ads = pe1-pe2-pe3

    pe = pe_ads
    x = np.arange(len(pe))
    
    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
    
    # Moving average
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")
    # fit_window = nsteps-1
    # Piecewise linear fits        
    slopes = []
    for start in range(0, len(pe) - fit_window + 1, fit_window):
        xs = x[start:start + fit_window]
        ys = pe[start:start + fit_window]
        z = np.polyfit(xs, ys, 1)
        p = np.poly1d(z)
        slopes.append(z[0])
        ax.plot(xs, p(xs), 'k--', lw=2)

    # Summary statistics        
    avg_slope = np.mean(slopes)
    final_slope = slopes[-1]
    # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

    slope_text = ", ".join([f"{s:.3f}" for s in slopes])
    # slope_text = ''
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads(kcal/mol)")
    ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 2000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# Settings
smooth_window = 1
frac = 0.20          # 20% of data used for each local regression

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    pe1 = np.asarray(results["close"]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    ax.plot(x, pe, color='red', alpha=0.35, lw=1, label='Raw trajectory')

    # LOWESS trend    
    trend = lowess(pe, x, frac=frac, return_sorted=True)

    ax.plot(trend[:,0], trend[:,1], 'k--', lw=3, label='LOWESS trend')

    # Overall slope of LOWESS trend
    slope, intercept = np.polyfit(trend[:,0], trend[:,1], 1)

    ax.set_title(f"({ratio})\nOverall slope = {slope:.5f}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 2000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# Settings
smooth_window = 1
frac = 0.20          # 20% of data used for each local regression

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    pe1 = np.asarray(results["close"]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    ax.plot(x, pe, color='red', alpha=0.35, lw=1, label='Raw trajectory')

    # LOWESS trend    
    trend = lowess(pe, x, frac=frac, return_sorted=True)

    ax.plot(trend[:,0], trend[:,1], 'k--', lw=3, label='LOWESS trend')

    # Overall slope of LOWESS trend
    slope, intercept = np.polyfit(trend[:,0], trend[:,1], 1)

    ax.set_title(f"({ratio})\nOverall slope = {slope:.5f}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# Settings
smooth_window = 1
frac = 0.20          # 20% of data used for each local regression

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(1, 1, figsize=(18, 6), sharex=True)

ax = axes
pe3 = np.asarray(results_31n["csh"]["PotEng"])
pe = pe3
x = np.arange(len(pe))

ax.plot(x, pe, color='red', alpha=0.35, lw=1, label='Raw trajectory')

# LOWESS trend    
trend = lowess(pe, x, frac=frac, return_sorted=True)

ax.plot(trend[:,0], trend[:,1], 'k--', lw=3, label='LOWESS trend')

# Overall slope of LOWESS trend
slope, intercept = np.polyfit(trend[:,0], trend[:,1], 1)

ax.set_title(f"({ratio})\nOverall slope = {slope:.5f}", fontsize=12)

ax.set_xlabel("Time (steps)")
ax.set_ylabel("E csh (kcal/mol)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase

# Plot
fig, axes = plt.subplots(1, 1, figsize=(18, 6), sharex=True)


ax = axes

pe3 = np.asarray(results_31n["csh"]["PotEng"])
pe = pe3
x = np.arange(len(pe))

# Raw trajectory
ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

# Moving average    
# pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
# x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
# ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)

# Sliding-window linear regression (continuous trend)    
half = fit_window // 2
trend_x = []
trend_y = []
slopes = []
for center in range(half, len(pe) - half, fit_step):
    xs = x[center-half:center+half]
    ys = pe[center-half:center+half]
    slope, intercept = np.polyfit(xs, ys, 1)
    slopes.append(slope)
    trend_x.append(center)
    trend_y.append(slope * center + intercept)

ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)

# Display representative slopes
n_display = 6
idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
ax.set_xlabel("Time (steps)")
ax.set_ylabel("E csh (kcal/mol)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

# Settings
smooth_window = 1
frac = 0.20          # 20% of data used for each local regression

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):
    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    pe1 = np.asarray(results["close"]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][:nsteps])
    pe = pe1 - pe2
    x = np.arange(len(pe))

    ax.plot(x, pe, color='red', alpha=0.35, lw=1, label='Raw trajectory')

    # LOWESS trend    
    trend = lowess(pe, x, frac=frac, return_sorted=True)

    ax.plot(trend[:,0], trend[:,1], 'k--', lw=3, label='LOWESS trend')

    # Overall slope of LOWESS trend
    slope, intercept = np.polyfit(trend[:,0], trend[:,1], 1)

    ax.set_title(f"({ratio})\nOverall slope = {slope:.5f}", fontsize=12)

    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("pe1-pe2 (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][:nsteps])
    pe = pe1 - pe2
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("pe1 - pe2 (kcal/mol)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run13_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log_re2.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log_re.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log_re2.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log_re.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log_re2.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log_re.lammps"
csh_log   = BASE / RUN / "S3/log_re.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][3500:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][3500:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # print average adsorption energy
    avg_ads_energy = np.mean(pe)
    print(f"Average adsorption energy for {ratio}: {avg_ads_energy:.3f} kcal/mol")

plt.tight_layout()
plt.show()



In [ ]:

# save adsorption energy as csv
pe_list  = []
for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][3500:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][3500:nsteps])
    pe = pe1 - pe2 - pe3
    pe_list.append(pe)


import pandas as pd
pd.DataFrame(pe_list[0]).to_csv("adsorption_energy_11.csv", index=False)
pd.DataFrame(pe_list[1]).to_csv("adsorption_energy_21.csv", index=False)
pd.DataFrame(pe_list[2]).to_csv("adsorption_energy_31.csv", index=False)


In [ ]:
pe_list[0][:1000].mean(), pe_list[1][:1000].mean(), pe_list[2][:1000].mean()

In [ ]:
pe_list[0][1000:2000].mean(), pe_list[1][1000:2000].mean(), pe_list[2][1000:2000].mean()

In [ ]:
pe_list[0][2000:3000].mean(), pe_list[1][2000:3000].mean(), pe_list[2][2000:3000].mean()

In [ ]:
pe_list[0][3000:4000].mean(), pe_list[1][3000:4000].mean(), pe_list[2][3000:4000].mean()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 500          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11, avg_peg21, avg_peg31]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps = len(results["close"]["PotEng"])
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][3000:nsteps])
    pe2_init = np.asarray(results["peg"]["PotEng"])
    # pe2 will be pe2_init and then append the average value for the rest of the steps from pe2_list[i]
    pe2 = np.concatenate((pe2_init, np.full(nsteps - len(pe2_init), pe2_list[i])))[3000:nsteps]
    pe3 = np.asarray(results["csh"]["PotEng"][3000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # set ylim -1000 to 1000
    ax.set_ylim(-500, 500)

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run13_pcff_merged"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log_merged.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log_merged.lammps"
csh_log   = BASE / RUN / "S3/log_merged.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log_merged.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log_merged.lammps"
csh_log   = BASE / RUN / "S3/log_merged.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log_merged.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log_merged.lammps"
csh_log   = BASE / RUN / "S3/log_merged.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


avg_peg11_ = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21_ = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31_ = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11_:.4f}, 2:1: {avg_peg21_:.4f}, 3:1: {avg_peg31_:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps1 = len(results["close"]["PotEng"])
    nsteps2 = len(results["csh"]["PotEng"])
    nsteps = min(nsteps1, nsteps2)
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][4000:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][4000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # print average adsorption energy
    avg_ads_energy = np.mean(pe)
    print(f"Average adsorption energy for {ratio}: {avg_ads_energy:.3f} kcal/mol")

plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 500          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]

    nsteps1 = len(results["close"]["PotEng"])
    nsteps2 = len(results["csh"]["PotEng"])
    nsteps = min(nsteps1, nsteps2)
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][3000:nsteps])
    pe2_init = np.asarray(results["peg"]["PotEng"])
    # pe2 will be pe2_init and then append the average value for the rest of the steps from pe2_list[i]
    pe2 = np.concatenate((pe2_init, np.full(nsteps - len(pe2_init), pe2_list[i])))[3000:nsteps]
    pe3 = np.asarray(results["csh"]["PotEng"][3000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
    # Moving average    
    pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # set ylim -1000 to 1000
    ax.set_ylim(-500, 500)

plt.tight_layout()
plt.show()

In [ ]:
RUN = "0_run13_pcff_merged_fix_max_steps"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log_merged.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log_merged.lammps"
csh_log   = BASE / RUN / "S3/log_merged.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log_merged.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log_merged.lammps"
csh_log   = BASE / RUN / "S3/log_merged.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log_merged.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log_merged.lammps"
csh_log   = BASE / RUN / "S3/log_merged.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


avg_peg11_ = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21_ = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31_ = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11_:.4f}, 2:1: {avg_peg21_:.4f}, 3:1: {avg_peg31_:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps1 = len(results["close"]["PotEng"])
    nsteps2 = len(results["csh"]["PotEng"])
    nsteps = min(nsteps1, nsteps2)
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][4000:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][4000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # print average adsorption energy
    avg_ads_energy = np.mean(pe)
    print(f"Average adsorption energy for {ratio}: {avg_ads_energy:.3f} kcal/mol")

plt.tight_layout()
plt.show()



In [ ]:
RUN = "0_run14_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/replica1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica1/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/replica1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica1/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


# ########### 3_1 ###############
# close_log = BASE / RUN / "3_1-6E/S1/replica1/log.lammps"
# peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
# csh_log   = BASE / RUN / "S3/replica1/log.lammps"
# results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
# plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


avg_peg11_ = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21_ = np.mean(results_21n['peg']['PotEng'][-1000:])
# avg_peg31_ = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11_:.4f}, 2:1: {avg_peg21_:.4f}, 3:1: ")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11_, avg_peg21_]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps1 = len(results["close"]["PotEng"])
    nsteps2 = len(results["csh"]["PotEng"])
    nsteps = min(nsteps1, nsteps2)
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][4000:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][4000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # print average adsorption energy
    avg_ads_energy = np.mean(pe)
    print(f"Average adsorption energy for {ratio}: {avg_ads_energy:.3f} kcal/mol")

plt.tight_layout()
plt.show()


In [ ]:
RUN = "0_run14_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/replica2/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica2/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/replica2/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica2/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/replica2/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica2/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


avg_peg11_ = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21_ = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31_ = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11_:.4f}, 2:1: {avg_peg21_:.4f}, 3:1: {avg_peg31_:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps1 = len(results["close"]["PotEng"])
    nsteps2 = len(results["csh"]["PotEng"])
    nsteps = min(nsteps1, nsteps2)
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][4000:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][4000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # print average adsorption energy
    avg_ads_energy = np.mean(pe)
    print(f"Average adsorption energy for {ratio}: {avg_ads_energy:.3f} kcal/mol")

plt.tight_layout()
plt.show()



In [ ]:
RUN = "0_run14_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/replica3/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica3/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/replica3/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica3/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/replica3/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
csh_log   = BASE / RUN / "S3/replica3/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


avg_peg11_ = np.mean(results_11n['peg']['PotEng'][-1000:])
avg_peg21_ = np.mean(results_21n['peg']['PotEng'][-1000:])
avg_peg31_ = np.mean(results_31n['peg']['PotEng'][-1000:])
print(f"Average PEG PotEng for 1:1: {avg_peg11_:.4f}, 2:1: {avg_peg21_:.4f}, 3:1: {avg_peg31_:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Settings
smooth_window = 100          # moving average window
fit_window = 1000          # size of local fitting window
fit_step = 50              # compute a new fit every 50 points

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]
pe2_list = [avg_peg11_, avg_peg21_, avg_peg31_]  # Average PotEng for PEG phase


# Plot
fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)

for i, (ratio, results) in enumerate(systems):

    ax = axes[i]
    nsteps1 = len(results["close"]["PotEng"])
    nsteps2 = len(results["csh"]["PotEng"])
    nsteps = min(nsteps1, nsteps2)
    
    # Adsorption energy
    pe1 = np.asarray(results["close"]["PotEng"][4000:nsteps])
    pe2 = pe2_list[i]
    pe3 = np.asarray(results["csh"]["PotEng"][4000:nsteps])
    pe = pe1 - pe2 - pe3
    x = np.arange(len(pe))

    # Raw trajectory
    ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw trajectory",)

    # Moving average    
    # pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid",)
    # x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
    # ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving average",)
    
    # Sliding-window linear regression (continuous trend)    
    half = fit_window // 2
    trend_x = []
    trend_y = []
    slopes = []
    for center in range(half, len(pe) - half, fit_step):
        xs = x[center-half:center+half]
        ys = pe[center-half:center+half]
        slope, intercept = np.polyfit(xs, ys, 1)
        slopes.append(slope)
        trend_x.append(center)
        trend_y.append(slope * center + intercept)

    ax.plot(trend_x, trend_y, "k--", lw=2.5, label="Sliding linear fit",)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)    
    # Display representative slopes
    n_display = 6
    idx = np.linspace(0, len(slopes)-1, n_display, dtype=int)
    slope_text = ", ".join(f"{slopes[j]:.3f}" for j in idx)
    ax.set_title(f"({ratio})\n{slope_text}", fontsize=12)
    ax.set_xlabel("Time (steps)")
    ax.set_ylabel("E ads (kcal/mol)")
    ax.grid(alpha=0.3)
    # print average adsorption energy
    avg_ads_energy = np.mean(pe)
    print(f"Average adsorption energy for {ratio}: {avg_ads_energy:.3f} kcal/mol")

plt.tight_layout()
plt.show()



In [ ]:
results_11n['peg']

In [ ]:
plt.plot(results_11n['close']['PotEng'][7000:])
plt.plot(results_11n['peg']['PotEng'][7000:])
plt.plot(results_11n['csh']['PotEng'][7000:])

In [ ]:
len(results_11n['close']['PotEng']), len(results_11n['peg']['PotEng']), len(results_11n['csh']['PotEng'])

In [ ]:
RUN = "0_run14_pcff"

########### S1 ###############
# 1_1
close_log = BASE / RUN / "1_1-12E/S1/replica1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S1/replica2/log.lammps"
csh_log   = BASE / RUN / "1_1-12E/S1/replica3/log.lammps"
results_s1_11 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


# 2_1
close_log = BASE / RUN / "2_1-8E/S1/replica1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S1/replica2/log.lammps"
csh_log   = BASE / RUN / "2_1-8E/S1/replica3/log.lammps"
results_s1_21 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


# 3_1
close_log = BASE / RUN / "3_1-6E/S1/replica1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S1/replica2/log.lammps"
csh_log   = BASE / RUN / "3_1-6E/S1/replica3/log.lammps"
results_s1_31 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)

########### S2 ###############
close_log = BASE / RUN / "1_1-12E/S2/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/log.lammps"
csh_log   = BASE / RUN / "3_1-6E/S2/log.lammps"
results_s2_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)


########### S3 ###############
close_log = BASE / RUN / "S3/replica1/log.lammps"
peg_log   = BASE / RUN / "S3/replica2/log.lammps"
csh_log   = BASE / RUN / "S3/replica3/log.lammps"
results_s3 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)






In [ ]:
print(len(results_s1_11['close']['PotEng']), len(results_s1_11['peg']['PotEng']), len(results_s1_11['csh']['PotEng']))
print(len(results_s1_21['close']['PotEng']), len(results_s1_21['peg']['PotEng']), len(results_s1_21['csh']['PotEng']))
print(len(results_s1_31['close']['PotEng']), len(results_s1_31['peg']['PotEng']), len(results_s1_31['csh']['PotEng']))
print(len(results_s2_11n['close']['PotEng']), len(results_s2_11n['peg']['PotEng']), len(results_s2_11n['csh']['PotEng']))
print(len(results_s3['close']['PotEng']), len(results_s3['peg']['PotEng']), len(results_s3['csh']['PotEng']))

In [ ]:
RUN = "0_run14_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "S3/replica1/log.lammps"
peg_log   = BASE / RUN / "S3/replica2/log.lammps"
csh_log   = BASE / RUN / "S3/replica3/log.lammps"
results_S3 = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=1000)

quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))

plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()
